# Airbnb Country Classification

## Project aim
Compare machine learning approaches for predicting the recorded COUNTRY
category from the available user characteristics.

## Target audience
A technical manager assessing whether the supplied user data supports
useful country classification.

## Research questions
1. How imbalanced are the country categories, and how does this affect
   model evaluation?
2. How do country proportions vary across recorded age and gender groups?
3. Can Logistic Regression and Random Forest outperform a simple
   majority-class baseline?

## Current limitations
The original dataset source, licence, and exact meanings of COUNTRY and
DATE_DIFF have not yet been confirmed. COUNTRY is treated as a recorded
category, without assuming that it represents a booking destination.
DATE_DIFF is excluded from modelling until its meaning and availability
at prediction time are verified.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from IPython.display import display
from google.colab import files

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import (
    cross_validate,
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

PROJECT = Path("/content/airbnb_project")
RAW = PROJECT / "data" / "raw"
RESULTS = PROJECT / "results"
FIGURES = RESULTS / "figures"

for folder in [RAW, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

print("Project folder:", PROJECT)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

In [ ]:
source_path = Path("/content/airbnb.csv")

if not source_path.exists():
    raise FileNotFoundError(
        "Upload airbnb.csv using the folder panel on the left first."
    )

raw_bytes = source_path.read_bytes()
raw_path = RAW / "airbnb.csv"
raw_path.write_bytes(raw_bytes)

df = pd.read_csv(raw_path)

expected_columns = {"ID", "DATE_DIFF", "GENDER", "AGE", "COUNTRY"}

if set(df.columns) != expected_columns:
    raise ValueError(
        f"Unexpected columns. Found: {list(df.columns)}"
    )

print("Dataset shape:", df.shape)
print("File SHA-256:", hashlib.sha256(raw_bytes).hexdigest())

display(df.head())
df.info()

In [ ]:
age_numeric = pd.to_numeric(df["AGE"], errors="coerce")

audit = {
    "rows": int(len(df)),
    "columns": list(df.columns),
    "missing_values": {
        col: int(value)
        for col, value in df.isna().sum().items()
    },
    "blank_strings": {
        col: int(
            df[col].astype("string").str.strip().eq("").sum()
        )
        for col in df.columns
    },
    "duplicate_rows": int(df.duplicated().sum()),
    "unique_ids": int(df["ID"].nunique()),
    "age_min": float(age_numeric.min()),
    "age_max": float(age_numeric.max()),
    "age_below_18": int((age_numeric < 18).sum()),
    "age_above_100": int((age_numeric > 100).sum()),
    "sha256": hashlib.sha256(raw_bytes).hexdigest(),
}

print(json.dumps(audit, indent=2))

(RESULTS / "data_audit.json").write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8",
)

if df["ID"].isna().any() or not df["ID"].is_unique:
    raise ValueError(
        "IDs are missing or repeated. Review grouping before splitting."
    )

if df["COUNTRY"].isna().any():
    raise ValueError("Missing target labels need investigation.")

## Experimental setup

COUNTRY is the provisional classification target.

ID is excluded because it identifies individual records rather than
providing a justified predictive characteristic.

DATE_DIFF is excluded because its meaning and availability at prediction
time have not been confirmed.

An 80/20 stratified split preserves approximate country proportions.
This assumes that each record represents an independent user. The split
would need reconsideration for a temporal or grouped deployment setting.

Detailed exploratory analysis and model selection use training data.
The test set is reserved for final evaluation.


In [ ]:
FEATURES = ["AGE", "GENDER"]
TARGET = "COUNTRY"

X = df[FEATURES].copy()
y = df[TARGET].astype(str)

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_df = df.iloc[train_idx].copy()

assert set(train_idx).isdisjoint(set(test_idx))
assert len(train_idx) + len(test_idx) == len(df)

split_manifest = pd.DataFrame({
    "ID": df["ID"],
    "split": "train",
})
split_manifest.loc[test_idx, "split"] = "test"
split_manifest.to_csv(RESULTS / "split_manifest.csv", index=False)

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Predictors:", FEATURES)
print("Target:", TARGET)

In [ ]:
country_summary = (
    y_train.value_counts()
    .rename_axis("country")
    .reset_index(name="count")
)

country_summary["percentage"] = (
    100 * country_summary["count"] / len(y_train)
)

display(country_summary.round(2))

country_summary.to_csv(
    RESULTS / "training_country_distribution.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=country_summary,
    x="country",
    y="percentage",
    color="#3874B5",
    ax=ax,
)
ax.set(
    title="Country distribution in the training set",
    xlabel="Recorded country category",
    ylabel="Percentage of training records",
)
fig.tight_layout()
fig.savefig(
    FIGURES / "01_country_distribution.png",
    dpi=200,
)
plt.show()

print(
    "Training majority-class baseline:",
    f"{country_summary.iloc[0]['percentage']:.2f}%"
)

In [ ]:
eda = train_df.copy()
eda["AGE_NUMERIC"] = pd.to_numeric(
    eda["AGE"], errors="coerce"
)

eda["AGE_GROUP"] = pd.cut(
    eda["AGE_NUMERIC"].where(
        eda["AGE_NUMERIC"].between(18, 100)
    ),
    bins=[18, 30, 45, 60, 101],
    labels=["18–29", "30–44", "45–59", "60–100"],
    right=False,
).astype("string").fillna("Outside range / missing")

gender_country = pd.crosstab(
    eda["GENDER"],
    eda["COUNTRY"],
    normalize="index",
) * 100

age_country = pd.crosstab(
    eda["AGE_GROUP"],
    eda["COUNTRY"],
    normalize="index",
) * 100

print("Group sizes:")
display(eda["GENDER"].value_counts())
display(eda["AGE_GROUP"].value_counts())

print("Country percentages within each gender category:")
display(gender_country.round(2))

print("Country percentages within each age group:")
display(age_country.round(2))

gender_country.to_csv(
    RESULTS / "country_percentages_by_gender.csv"
)
age_country.to_csv(
    RESULTS / "country_percentages_by_age_group.csv"
)

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

sns.heatmap(
    gender_country,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Percentage within group"},
)
axes[0].set(
    title="Country percentages within recorded gender categories",
    xlabel="Recorded country",
    ylabel="Recorded gender",
)

sns.heatmap(
    age_country,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    ax=axes[1],
    cbar_kws={"label": "Percentage within group"},
)
axes[1].set(
    title="Country percentages within age groups",
    xlabel="Recorded country",
    ylabel="Age group",
)

fig.tight_layout()
fig.savefig(FIGURES / "03_group_country_percentages.png", dpi=200)
plt.show()

## Preprocessing

Ages outside the provisional 18–100 range are converted to missing values.
Median age imputation is learned from each training fold, with an indicator
for missing or invalid age.

Gender is imputed if necessary and one-hot encoded.

Age is standardised for Logistic Regression. Scaling is unnecessary for
Random Forest.

All learned preprocessing occurs inside pipelines to prevent information
from validation or test records entering the fitting process.

In [ ]:
class AgeRangeCleaner(BaseEstimator, TransformerMixin):
    """Apply a fixed, provisional age-validity rule."""

    def __init__(self, minimum=18, maximum=100):
        self.minimum = minimum
        self.maximum = maximum

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        values = (
            pd.DataFrame(X)
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=float)
        )

        invalid = (
            (values < self.minimum)
            | (values > self.maximum)
        )

        values[invalid] = np.nan
        return values


def make_preprocessor(scale_age=False):
    age_steps = [
        ("validity", AgeRangeCleaner()),
        (
            "impute",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]

    if scale_age:
        age_steps.append(("scale", StandardScaler()))

    gender_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        (
            "encode",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ])

    return ColumnTransformer([
        ("age", Pipeline(age_steps), ["AGE"]),
        ("gender", gender_pipeline, ["GENDER"]),
    ])


models = {
    "Majority baseline": Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "Logistic Regression": Pipeline([
        ("preprocess", make_preprocessor(scale_age=True)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=SEED,
            ),
        ),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", make_preprocessor()),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=5,
                random_state=SEED,
                n_jobs=1,
            ),
        ),
    ]),
}

print("Models ready:", list(models))

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
}

cv_rows = []

for name, model in models.items():
    print("Evaluating:", name)

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
        error_score="raise",
    )

    row = {"model": name}

    for metric in scoring:
        row[f"{metric}_mean"] = scores[
            f"test_{metric}"
        ].mean()

        row[f"{metric}_std"] = scores[
            f"test_{metric}"
        ].std(ddof=1)

    row["training_macro_f1_mean"] = scores[
        "train_macro_f1"
    ].mean()

    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values(
    "macro_f1_mean",
    ascending=False,
)

display(cv_results.round(4))

cv_results.to_csv(
    RESULTS / "initial_cross_validation_results.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    cv_results["model"],
    cv_results["macro_f1_mean"],
    yerr=cv_results["macro_f1_std"],
    capsize=5,
    color="#3874B5",
)
ax.set(
    title="Initial model comparison using five-fold cross-validation",
    xlabel="Model",
    ylabel="Mean macro-F1 ± one fold standard deviation",
    ylim=(0, 1),
)
fig.tight_layout()
fig.savefig(FIGURES / "04_initial_model_comparison.png", dpi=200)
plt.show()

In [ ]:
parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "model__max_depth": [6, None],
        "model__min_samples_leaf": [5, 20],
        "model__class_weight": [None, "balanced"],
    },
}

searches = {}
tuning_rows = []

for name, parameter_grid in parameter_grids.items():
    print("Tuning:", name)

    search = GridSearchCV(
        estimator=models[name],
        param_grid=parameter_grid,
        scoring=scoring,
        refit="macro_f1",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
    )

    search.fit(X_train, y_train)
    searches[name] = search

    best_index = search.best_index_

    tuning_rows.append({
        "model": name,
        "best_cv_macro_f1": search.best_score_,
        "cv_macro_f1_std": search.cv_results_[
            "std_test_macro_f1"
        ][best_index],
        "best_parameters": json.dumps(search.best_params_),
    })

    pd.DataFrame(search.cv_results_).to_csv(
        RESULTS / (
            name.lower().replace(" ", "_")
            + "_search_results.csv"
        ),
        index=False,
    )

    print("Best parameters:", search.best_params_)
    print("Best CV macro-F1:", round(search.best_score_, 4))

tuning_summary = pd.DataFrame(tuning_rows).sort_values(
    "best_cv_macro_f1",
    ascending=False,
)

display(tuning_summary)

tuning_summary.to_csv(
    RESULTS / "tuning_summary.csv",
    index=False,
)

In [ ]:
baseline_cv = float(
    cv_results.loc[
        cv_results["model"] == "Majority baseline",
        "macro_f1_mean",
    ].iloc[0]
)

selection_scores = {
    "Majority baseline": baseline_cv,
    **{
        name: search.best_score_
        for name, search in searches.items()
    },
}

selected_name = max(
    selection_scores,
    key=selection_scores.get,
)

selection_record = {
    "selected_model": selected_name,
    "selection_metric": "mean training CV macro-F1",
    "selection_scores": {
        name: float(score)
        for name, score in selection_scores.items()
    },
    "features": FEATURES,
    "excluded_features": {
        "ID": "Unique identifier",
        "DATE_DIFF": "Meaning and prediction-time availability unverified",
    },
    "age_rule": "Outside 18–100 treated as missing; provisional",
    "random_seed": SEED,
    "test_fraction": 0.20,
}

(RESULTS / "model_selection.json").write_text(
    json.dumps(selection_record, indent=2),
    encoding="utf-8",
)

print("Selected before viewing test results:", selected_name)
print(json.dumps(selection_record, indent=2))

In [ ]:
baseline = models["Majority baseline"]
baseline.fit(X_train, y_train)

final_models = {
    "Majority baseline": baseline,
    **{
        name: search.best_estimator_
        for name, search in searches.items()
    },
}

test_rows = []
predictions = {}
labels = sorted(y_train.unique())

for name, model in final_models.items():
    predicted = model.predict(X_test)
    predictions[name] = predicted

    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predicted),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, predicted
        ),
        "macro_f1": f1_score(
            y_test,
            predicted,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_test,
            predicted,
            average="weighted",
            zero_division=0,
        ),
    })

    report = classification_report(
        y_test,
        predicted,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(report).T.to_csv(
        RESULTS / (
            name.lower().replace(" ", "_")
            + "_test_classification_report.csv"
        )
    )

test_results = pd.DataFrame(test_rows)

display(test_results.round(4))

test_results.to_csv(
    RESULTS / "final_test_results.csv",
    index=False,
)

print("Previously selected model:", selected_name)

In [ ]:
selected_predictions = predictions[selected_name]

print(
    classification_report(
        y_test,
        selected_predictions,
        labels=labels,
        zero_division=0,
    )
)

fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    selected_predictions,
    labels=labels,
    normalize="true",
    values_format=".2f",
    cmap="Blues",
    ax=ax,
    colorbar=False,
)

ax.set_title(
    f"Normalised test confusion matrix: {selected_name}"
)

fig.tight_layout()
fig.savefig(
    FIGURES / "05_selected_model_confusion_matrix.png",
    dpi=200,
)
plt.show()

prediction_output = pd.DataFrame({
    "ID": df.iloc[test_idx]["ID"].to_numpy(),
    "actual_country": y_test.to_numpy(),
    "predicted_country": selected_predictions,
})

prediction_output.to_csv(
    RESULTS / "selected_model_test_predictions.csv",
    index=False,
)

print("Examples of incorrect predictions:")
display(
    prediction_output[
        prediction_output["actual_country"]
        != prediction_output["predicted_country"]
    ].head(15)
)

In [ ]:
plot_data = test_results.melt(
    id_vars="model",
    value_vars=[
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
    ],
    var_name="metric",
    value_name="score",
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(
    data=plot_data,
    x="model",
    y="score",
    hue="metric",
    ax=ax,
)

ax.set(
    title="Held-out test performance",
    xlabel="Model",
    ylabel="Score",
    ylim=(0, 1),
)

ax.legend(title="Metric", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(FIGURES / "06_final_model_comparison.png", dpi=200)
plt.show()

In [ ]:
environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "seed": SEED,
    "dataset_sha256": hashlib.sha256(raw_bytes).hexdigest(),
}

(RESULTS / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

requirements = "\n".join([
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"matplotlib=={matplotlib.__version__}",
    f"seaborn=={sns.__version__}",
]) + "\n"

(PROJECT / "requirements.txt").write_text(
    requirements,
    encoding="utf-8",
)

notes = """
AIRBNB PRACTICE PROJECT — INITIAL EXPERIMENT

Run the accompanying Colab notebook from top to bottom.
Upload the original airbnb.csv when prompted.

Predictors: AGE and GENDER.
Target: recorded COUNTRY category.
ID excluded as an identifier.
DATE_DIFF excluded pending definition and leakage review.

Age outside 18–100 is provisionally treated as missing.
Median imputation and encoding are fitted inside training pipelines.
Selection uses stratified five-fold training cross-validation.
A fixed stratified 20% test set is reserved for final evaluation.

Original source, licence, COUNTRY meaning, and DATE_DIFF meaning
remain unverified. Do not claim a confirmed booking destination task.
Do not publicly redistribute raw data until permission is established.

This package contains results and environment details.
The notebook must be downloaded separately and retained with it.
"""

(PROJECT / "RUN_NOTES.txt").write_text(
    notes.strip() + "\n",
    encoding="utf-8",
)

# Package results without redistributing the raw dataset.
export_folder = Path("/content/airbnb_results_package")

if export_folder.exists():
    shutil.rmtree(export_folder)

export_folder.mkdir()
shutil.copytree(RESULTS, export_folder / "results")
shutil.copy(PROJECT / "requirements.txt", export_folder)
shutil.copy(PROJECT / "RUN_NOTES.txt", export_folder)

zip_path = shutil.make_archive(
    "/content/airbnb_results",
    "zip",
    root_dir=export_folder,
)

files.download(zip_path)

In [ ]:
training_age = pd.to_numeric(
    train_df["AGE"], errors="coerce"
)

age_summary = pd.Series({
    "training_records": len(training_age),
    "missing_or_non_numeric": training_age.isna().sum(),
    "below_18": training_age.lt(18).sum(),
    "above_100": training_age.gt(100).sum(),
    "minimum": training_age.min(),
    "median": training_age.median(),
    "maximum": training_age.max(),
})

display(age_summary)
age_summary.to_csv(RESULTS / "training_age_summary.csv")

# Provisional range: retain all records, but show plausible ages separately.
plausible_age = training_age.where(
    training_age.between(18, 100)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    plausible_age.dropna(),
    bins=30,
    color="#3874B5",
    ax=axes[0],
)
axes[0].set(
    title="Training ages within the provisional 18–100 range",
    xlabel="Recorded age",
    ylabel="Number of records",
)

sns.boxplot(
    x=plausible_age.dropna(),
    color="#8EB7DA",
    ax=axes[1],
)
axes[1].set(
    title="Training age distribution within the provisional range",
    xlabel="Recorded age",
)

fig.tight_layout()
fig.savefig(
    FIGURES / "02_age_distribution.png",
    dpi=200,
)
plt.show()

print("Most frequent recorded ages above 100:")
display(
    training_age[training_age > 100]
    .value_counts()
    .head(15)
)

### Interpretation of age quality

The supplied dataset contains implausible recorded ages, including a
maximum value of 2014. These values cannot safely be interpreted as
birth years without documentation.

For this initial experiment, ages outside 18–100 were treated as missing.
The records were retained, and missing ages were imputed using the median
learned within each training fold.

The 18–100 range is a provisional modelling assumption rather than a
verified rule from the dataset publisher. It may also flag genuine older
users. Source documentation and sensitivity analysis would be needed to
justify a final cleaning policy.

## Findings from the initial experiment

### Data quality

The supplied dataset contains 50,098 records and five columns.
No missing cells or exact duplicate rows were detected, and IDs were
unique. However, 716 recorded ages exceeded 100 and 16 were below 18.

Ages outside the provisional 18–100 range were treated as missing.
Median imputation was fitted inside the training pipelines, retaining
records while avoiding direct use of implausible age values.
The validity of this range remains a limitation.

### Exploratory analysis

US was the largest country category and PT was the smallest.
Approximately 79.5% of records belonged to US, creating substantial
class imbalance.

Country percentages within groups were used to compare age and gender
categories because raw counts would be affected by unequal group sizes.
These comparisons describe associations and do not establish causation.

### Model comparison

The majority baseline, Logistic Regression, and Random Forest obtained
the same mean cross-validation macro-F1 of approximately 0.0886.

The selected Logistic Regression settings were C=0.1 and no class
weighting. The selected Random Forest used 200 trees, maximum depth 6,
minimum samples per leaf 5, and no class weighting.

Class weighting was tested, but it did not produce a higher best
cross-validation macro-F1 than the selected unweighted configurations.

### Final evaluation

All three evaluated models predicted US for every test record.
Each achieved accuracy of 79.54%, balanced accuracy of 10.00%, and
macro-F1 of approximately 0.0886.

Recall was zero for every non-US country. Consequently, the accuracy
score concealed the models' failure to distinguish minority categories.

The selection code retained the majority baseline because all three
cross-validation selection scores tied and the baseline appeared first.
Neither trained algorithm demonstrated an improvement over that baseline.

### Limitations

The models used only AGE and GENDER. DATE_DIFF was excluded because its
meaning and availability at prediction time remained unverified.
The exact meaning of COUNTRY and the original dataset licence also
remain unresolved.

The random stratified split assumes independent records and does not
establish performance on future users or other populations.
The results do not show that all possible Airbnb prediction approaches
would fail; they describe these models, predictors, and experimental
settings.

In [ ]:
age_export = Path("/content/airbnb_age_analysis")
age_export.mkdir(exist_ok=True)

shutil.copy(
    RESULTS / "training_age_summary.csv",
    age_export / "training_age_summary.csv",
)
shutil.copy(
    FIGURES / "02_age_distribution.png",
    age_export / "02_age_distribution.png",
)

age_zip = shutil.make_archive(
    "/content/airbnb_age_analysis",
    "zip",
    root_dir=age_export,
)

files.download(age_zip)